# 🎤 Assistente de Análise de Dados por Voz
Projeto integrando captura de áudio, Whisper, GPT e análise com Pandas.
---

## 1️⃣ Instalação de Bibliotecas

In [ ]:
!pip install openai
!pip install openai-whisper
!pip install gtts
!pip install pandas
!pip install matplotlib

## 2️⃣ Importações Básicas

In [ ]:
import os
import whisper
import pandas as pd
import matplotlib.pyplot as plt
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output, files
from base64 import b64decode
from openai import OpenAI
from getpass import getpass

## 3️⃣ Inserir API Key com Segurança

In [ ]:
os.environ['OPENAI_API_KEY'] = getpass('Digite sua API Key: ')
client = OpenAI()

## 4️⃣ Upload e Carregamento do Dataset

In [ ]:
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]
df = pd.read_csv(nome_arquivo)
display(df.head())

## 5️⃣ Gravação de Áudio 🎤

In [ ]:
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])
    file_name = 'audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)
    return file_name

print('Ouvindo...')
audio_file = record(5)
display(Audio(audio_file))

## 6️⃣ Transcrição com Whisper

In [ ]:
model = whisper.load_model('base')
result = model.transcribe(audio_file)
texto_pergunta = result['text']
print('Pergunta:', texto_pergunta)

## 7️⃣ Geração de Código com GPT

In [ ]:
prompt = f"""
Você é um analista de dados.

Temos um dataframe chamado df com as seguintes colunas:
{df.columns.tolist()}

Aqui estão as primeiras linhas:
{df.head().to_string()}

Gere apenas código Python usando pandas e matplotlib (se necessário).
Regras:
- Use apenas o dataframe chamado df
- Se for pedido gráfico, use matplotlib
- Sempre imprima o resultado usando print()
- Não explique nada
- Retorne apenas código executável

Pergunta: {texto_pergunta}
"""

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': 'Você é especialista em análise de dados.'},
        {'role': 'user', 'content': prompt}
    ]
)

codigo = response.choices[0].message.content
print('Código gerado:')
print(codigo)

## 8️⃣ Execução da Análise

In [ ]:
try:
    exec(codigo)
    resposta_texto = 'Análise concluída com sucesso.'
except Exception as e:
    resposta_texto = f'Erro na execução: {str(e)}'
    print(resposta_texto)

## 9️⃣ Conversão da Resposta em Voz 🔊

In [ ]:
tts = gTTS(resposta_texto, lang='pt')
tts.save('resposta.mp3')
Audio('resposta.mp3')